# 01 — Data Collection & Cleaning

**Project:** Where Should Amazon Air Build Its Next Hub?
**Author:** Aikerim Imanbayeva

## What this notebook does

1. Loads all five raw data sources from `data/raw/`
2. Standardizes column names, types, and ID formats
3. Defines the candidate airport universe (US + operational + public-use + runway ≥ 7,000 ft + CONUS/HI)
4. Computes airport-level cargo summaries from BTS T-100
5. Identifies the top ~80 cargo airports (so we know which NOAA weather stations to pull later)
6. Saves cleaned outputs to `data/processed/`

## Output files (saved to `data/processed/`)

- `candidate_airports.csv` — master table of ~494 viable candidate airports
- `cargo_summary_2025.csv` — airport-level cargo activity for 2025 (the scoring year)
- `cargo_summary_yearly.csv` — airport-level cargo time series 2020–2025
- `top_cargo_airports.csv` — top 80 by 2025 cargo (input for NOAA scoping)
- `county_population_with_centroids.csv` — county-level population with centroid lat/lon

## Setup

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

# Resolve project paths from the notebook's location
PROJECT_ROOT = Path('..').resolve()
RAW = PROJECT_ROOT / 'data' / 'raw'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'RAW          = {RAW}')
print(f'PROCESSED    = {PROCESSED}')

---
## 1. BTS T-100 Segment — 6-year cargo activity

Concatenate the six annual CSVs into a single DataFrame. We have `YEAR` already as a column, so we can identify each row's year after concatenation.

Universe filter applied to T-100:
- US domestic segments only (`ORIGIN_COUNTRY == 'US'` AND `DEST_COUNTRY == 'US'`)
- Cargo-relevant aircraft configuration: filter `AIRCRAFT_CONFIG` to freighter codes

**`AIRCRAFT_CONFIG` codes** (BTS reference):
- `1` = Passenger
- `2` = Cargo (freighter)
- `3` = Passenger/Cargo combined
- `4` = Other/Unknown

For cargo analysis, we keep configs `2` and `3`.

In [ ]:
bts_files = sorted((RAW / 'bts_t100').glob('t100_segment_*.csv'))
print(f'Found {len(bts_files)} annual BTS files')
for f in bts_files:
    print(f'  {f.name}')

# Read and concatenate
t100_raw = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in bts_files],
    ignore_index=True,
)
print(f'\nTotal T-100 rows (all years): {len(t100_raw):,}')
print(f'Years present: {sorted(t100_raw["YEAR"].unique())}')

In [ ]:
# Filter to US domestic cargo-relevant flights
t100 = t100_raw[
    (t100_raw['ORIGIN_COUNTRY'] == 'US') &
    (t100_raw['DEST_COUNTRY'] == 'US') &
    (t100_raw['AIRCRAFT_CONFIG'].isin([2, 3]))
].copy()

# Compute total cargo (freight + mail) per row
t100['TOTAL_CARGO_LB'] = t100['FREIGHT'].fillna(0) + t100['MAIL'].fillna(0)

print(f'After filtering to US domestic cargo segments: {len(t100):,} rows')
print(f'\nTotal cargo by year (millions of lbs):')
print((t100.groupby('YEAR')['TOTAL_CARGO_LB'].sum() / 1e6).round(1).to_string())

---
## 2. FAA Airport Master Record

Load the airports and runways CSVs. Convert FAA's DMS coordinates (e.g., `39-02-55.8150N`) to decimal degrees. Join runways to airports via the `AIRPORT_ID` (in runways) → `GLOBAL_ID` (in airports) UUID.

In [ ]:
def parse_dms(coord_str):
    """Convert FAA DMS string like '39-02-55.8150N' to decimal degrees."""
    if pd.isna(coord_str):
        return np.nan
    m = re.match(r'(\d+)-(\d+)-([\d.]+)([NSEW])', str(coord_str).strip())
    if not m:
        return np.nan
    deg, minute, second, direction = m.groups()
    dd = float(deg) + float(minute)/60 + float(second)/3600
    if direction in ('S', 'W'):
        dd = -dd
    return dd

faa_ap_raw = pd.read_csv(RAW / 'faa' / 'faa_airports.csv', low_memory=False)
faa_rw_raw = pd.read_csv(RAW / 'faa' / 'faa_runways.csv', low_memory=False)

print(f'FAA airports: {len(faa_ap_raw):,} rows')
print(f'FAA runways:  {len(faa_rw_raw):,} rows')

# Convert coordinates
faa_ap = faa_ap_raw.copy()
faa_ap['LAT'] = faa_ap['LATITUDE'].apply(parse_dms)
faa_ap['LON'] = faa_ap['LONGITUDE'].apply(parse_dms)
print(f'\nDMS conversion check — CVG should be ~(39.05, -84.67):')
print(faa_ap[faa_ap['IDENT'] == 'CVG'][['IDENT', 'NAME', 'LAT', 'LON']].to_string(index=False))

---
## 3. Candidate airport universe

Apply the defensible scoping filter:
1. Country = United States
2. Operational status = OPERATIONAL
3. Public use (`PRIVATEUSE == 0`)
4. Type = AD (airport, not heliport/seaplane/glider)
5. At least one runway ≥ 7,000 ft
6. CONUS + Hawaii only (exclude AK, PR, VI, GU, AS, MP)

Interview defense: *"I scoped to airports that can physically operate the 767-300F that Amazon Air uses, in the geography Amazon's network covers."*

In [ ]:
# Compute per-airport runway summary first
rw_summary = (
    faa_rw_raw
    .groupby('AIRPORT_ID')
    .agg(
        n_runways=('DESIGNATOR', 'count'),
        longest_runway_ft=('LENGTH', 'max'),
        n_runways_over_7000ft=('LENGTH', lambda s: (s >= 7000).sum()),
    )
    .reset_index()
)

# Apply the candidate filter on airports
ap_clean = faa_ap[
    (faa_ap['COUNTRY'] == 'UNITED STATES') &
    (faa_ap['OPERSTATUS'] == 'OPERATIONAL') &
    (faa_ap['PRIVATEUSE'] == 0) &
    (faa_ap['TYPE_CODE'] == 'AD') &
    (~faa_ap['STATE'].isin(['AK', 'PR', 'VI', 'GU', 'AS', 'MP']))
].copy()

# Join runway summary (note: runways.AIRPORT_ID == airports.GLOBAL_ID, both UUIDs)
candidates = ap_clean.merge(
    rw_summary, left_on='GLOBAL_ID', right_on='AIRPORT_ID', how='left'
)

# Apply runway length filter
candidates = candidates[candidates['longest_runway_ft'] >= 7000].copy()

# Keep only the columns we'll need downstream
keep_cols = ['IDENT', 'ICAO_ID', 'NAME', 'SERVCITY', 'STATE', 'LAT', 'LON',
             'ELEVATION', 'n_runways', 'longest_runway_ft', 'n_runways_over_7000ft',
             'GLOBAL_ID']
candidates = candidates[keep_cols].reset_index(drop=True)
candidates.columns = [c.lower() for c in candidates.columns]

print(f'Candidate airport universe: {len(candidates):,} airports')
print(f'\nTop 10 by longest runway:')
print(candidates.nlargest(10, 'longest_runway_ft')[['ident', 'name', 'state', 'longest_runway_ft']].to_string(index=False))

---
## 4. Census ACS population + TIGER county polygons

Load county population, join to county polygons via FIPS, and compute centroid lat/lon for each county. These centroids will be used in the cleaning notebook's downstream catchment computation.

In [ ]:
# Census population — skip the first label row
pop_raw = pd.read_csv(RAW / 'census' / 'acs_county_population_2023.csv', skiprows=[1], low_memory=False)

# Extract 5-digit FIPS county code from GEO_ID (format: '0500000US01001')
pop = pop_raw[['GEO_ID', 'NAME', 'B01003_001E']].copy()
pop.columns = ['geo_id', 'county_name', 'population']
pop['fips'] = pop['geo_id'].str[-5:]
pop['population'] = pd.to_numeric(pop['population'], errors='coerce')
pop = pop[['fips', 'county_name', 'population']]

print(f'County population rows: {len(pop):,}')
print(f'Total US population: {pop["population"].sum():,.0f}')

In [ ]:
# County polygons from TIGER zip
counties = gpd.read_file(f"zip://{RAW / 'census' / 'tl_2024_us_county.zip'}")
print(f'County polygons: {len(counties):,}')
print(f'CRS: {counties.crs}')

# Compute centroids in a projected CRS (USA Contiguous Albers Equal Area) then transform back to WGS84
counties_proj = counties.to_crs(epsg=5070)
centroids = counties_proj.geometry.centroid.to_crs(epsg=4326)
counties['centroid_lat'] = centroids.y
counties['centroid_lon'] = centroids.x

# Join with population on FIPS (GEOID in TIGER = state(2) + county(3) = 5-digit FIPS)
counties_pop = counties[['GEOID', 'NAME', 'STATEFP', 'centroid_lat', 'centroid_lon']].merge(
    pop, left_on='GEOID', right_on='fips', how='inner'
)
counties_pop.columns = [c.lower() for c in counties_pop.columns]

print(f'\nCounty polygons joined to population: {len(counties_pop):,}')
print(f'\nSample (5 random rows):')
print(counties_pop.sample(5, random_state=42).to_string(index=False))

---
## 5. Amazon Air facility list

Load the hand-built CSV. This is small and clean — just load it.

In [ ]:
amazon = pd.read_csv(RAW / 'amazon_air' / 'amazon_air_facilities.csv')
print(f'Amazon Air facilities: {len(amazon)}')
print(f'\nRole breakdown:')
print(amazon['role'].value_counts().to_string())
amazon.head()

---
## 6. Airport-level cargo summaries

Aggregate T-100 to airport level. Two views:
1. **By origin** — cargo *leaving* the airport (lifted)
2. **By destination** — cargo *arriving* at the airport (received)

Total throughput = origin lift + destination receipt. We'll compute both.

In [ ]:
# Cargo by origin
by_origin = (
    t100.groupby(['YEAR', 'ORIGIN'])
    .agg(
        cargo_lifted_lb=('TOTAL_CARGO_LB', 'sum'),
        departures=('DEPARTURES_PERFORMED', 'sum'),
    )
    .reset_index()
    .rename(columns={'ORIGIN': 'airport_id'})
)

# Cargo by destination
by_dest = (
    t100.groupby(['YEAR', 'DEST'])
    .agg(
        cargo_received_lb=('TOTAL_CARGO_LB', 'sum'),
        arrivals=('DEPARTURES_PERFORMED', 'sum'),
    )
    .reset_index()
    .rename(columns={'DEST': 'airport_id'})
)

# Outer-join the two views
cargo_yearly = by_origin.merge(
    by_dest, on=['YEAR', 'airport_id'], how='outer'
).fillna(0)
cargo_yearly['total_throughput_lb'] = (
    cargo_yearly['cargo_lifted_lb'] + cargo_yearly['cargo_received_lb']
)
cargo_yearly['total_operations'] = (
    cargo_yearly['departures'] + cargo_yearly['arrivals']
)
cargo_yearly.columns = [c.lower() for c in cargo_yearly.columns]

print(f'Cargo summary rows (airport × year): {len(cargo_yearly):,}')
print(f'Distinct airports with cargo activity: {cargo_yearly["airport_id"].nunique():,}')

In [ ]:
# Cross-sectional view for the scoring year (2025)
cargo_2025 = cargo_yearly[cargo_yearly['year'] == 2025].copy().drop(columns='year')

print(f'2025 cargo summary: {len(cargo_2025):,} airports')
print(f'\nTop 15 US cargo airports by 2025 throughput (millions of lbs):')
top15 = (
    cargo_2025
    .nlargest(15, 'total_throughput_lb')
    .assign(throughput_M_lbs=lambda d: (d['total_throughput_lb']/1e6).round(1))
    [['airport_id', 'throughput_M_lbs', 'total_operations']]
)
print(top15.to_string(index=False))

---
## 7. Top-cargo airports (input for NOAA scoping)

Identify the top 80 candidate airports by 2025 cargo throughput. We'll pull NOAA weather data for ONLY these airports rather than the full 494 candidate universe — a defensible scoping decision.

In [ ]:
# Join 2025 cargo data with the candidate universe
candidates_with_cargo = candidates.merge(
    cargo_2025, left_on='ident', right_on='airport_id', how='left'
).fillna({'cargo_lifted_lb': 0, 'cargo_received_lb': 0,
          'total_throughput_lb': 0, 'departures': 0,
          'arrivals': 0, 'total_operations': 0})

# Rank by cargo throughput
candidates_with_cargo = candidates_with_cargo.sort_values(
    'total_throughput_lb', ascending=False
).reset_index(drop=True)
candidates_with_cargo['cargo_rank_2025'] = candidates_with_cargo.index + 1

# Top 80 for NOAA scope
top80 = candidates_with_cargo.head(80).copy()

print(f'Top 80 candidates (these will receive NOAA weather data):')
print(top80[['cargo_rank_2025', 'ident', 'name', 'state',
             'longest_runway_ft', 'total_throughput_lb']].head(25).to_string(index=False))

# Mark Amazon Air existing facilities
top80['is_amazon_air'] = top80['ident'].isin(amazon['airport_id'])
print(f'\nAmazon Air presence among top 80: {top80["is_amazon_air"].sum()} airports')

---
## 8. Save processed outputs

In [ ]:
# Save all processed files
candidates.to_csv(PROCESSED / 'candidate_airports.csv', index=False)
cargo_2025.to_csv(PROCESSED / 'cargo_summary_2025.csv', index=False)
cargo_yearly.to_csv(PROCESSED / 'cargo_summary_yearly.csv', index=False)
top80.to_csv(PROCESSED / 'top_cargo_airports.csv', index=False)
counties_pop.to_csv(PROCESSED / 'county_population_with_centroids.csv', index=False)

print('Saved to data/processed/:')
for f in sorted(PROCESSED.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:50s}  {size_kb:>8.1f} KB')

---
## Summary

**What this notebook produced:**

| File | What it contains | Used by |
| --- | --- | --- |
| `candidate_airports.csv` | 494 airports passing the universe filter | All downstream notebooks |
| `cargo_summary_2025.csv` | 2025 cargo activity per airport | `03_hub_scoring.ipynb` |
| `cargo_summary_yearly.csv` | Cargo time series 2020–2025 | `02_eda.ipynb`, future projects |
| `top_cargo_airports.csv` | Top 80 with all attributes | NOAA scope, EDA |
| `county_population_with_centroids.csv` | County pop + centroid lat/lon | `03_hub_scoring.ipynb` catchment calc |

**Next steps:**

1. Build `02_eda.ipynb` — explore cargo flow distribution, geographic patterns, year-over-year trends
2. Pull NOAA weather data for the 80 airports identified here
3. Build `03_hub_scoring.ipynb` — construct the Hub Fitness Index